# Data Quality Analysis

**Purpose**: Profile data quality, identify issues, and assess data trustworthiness

**Data Quality Dimensions**:
- **Completeness**: Are all required fields populated?
- **Accuracy**: Are values correct and valid?
- **Consistency**: Do related values agree?
- **Uniqueness**: Are duplicates present?
- **Timeliness**: Is data current and up-to-date?
- **Integrity**: Are relationships maintained?

**Business Impact**:
- Poor data quality → Bad decisions
- Incomplete records → Lost revenue
- Duplicates → Customer experience issues
- Invalid data → Compliance risks

**Target Audience**: Chief Data Officer, Data Engineering, Data Governance

In [ ]:
# Import libraries
import sys
sys.path.append('/home/jovyan/work')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import re
import warnings
warnings.filterwarnings('ignore')

from middleware import get_db, query, get_table_df, DataQualityChecker

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")
%matplotlib inline

print("✓ Libraries imported successfully")

## 1. Database Overview & Profiling

In [ ]:
# Connect and get table inventory
db = get_db()
tables = db.list_tables()

print("DATABASE OVERVIEW")
print("=" * 70)
print(f"Database: {db.database}")
print(f"Total Tables: {len(tables)}\n")

# Get row counts for all tables
table_stats = []
for table in tables:
    count = db.get_row_count(table)
    table_stats.append({'Table': table, 'Row Count': count})

stats_df = pd.DataFrame(table_stats).sort_values('Row Count', ascending=False)
print("TABLE SIZES:")
print(stats_df.to_string(index=False))
print("=" * 70)

## 2. Completeness Analysis

In [ ]:
# Profile main tables for null values
def analyze_completeness(table_name):
    """Analyze null/missing values in a table."""
    df = get_table_df(table_name)
    
    total_rows = len(df)
    null_counts = df.isnull().sum()
    null_percentages = (null_counts / total_rows * 100).sort_values(ascending=False)
    
    completeness_df = pd.DataFrame({
        'Column': null_percentages.index,
        'Null Count': null_counts[null_percentages.index].values,
        'Null %': null_percentages.values,
        'Completeness %': 100 - null_percentages.values
    })
    
    return completeness_df

# Analyze key tables
print("COMPLETENESS ANALYSIS - CUSTOMERS TABLE")
print("=" * 70)
customers_completeness = analyze_completeness('customers')
print(customers_completeness[customers_completeness['Null %'] > 0].to_string(index=False))

print("\n" + "=" * 70)
print("COMPLETENESS ANALYSIS - TRANSACTIONS TABLE")
print("=" * 70)
transactions_completeness = analyze_completeness('transactions')
print(transactions_completeness[transactions_completeness['Null %'] > 0].to_string(index=False))

In [ ]:
# Visualize completeness
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Customers
top_missing_customers = customers_completeness.nlargest(10, 'Null %')
axes[0].barh(range(len(top_missing_customers)), top_missing_customers['Null %'], color='#D72638', alpha=0.7)
axes[0].set_yticks(range(len(top_missing_customers)))
axes[0].set_yticklabels(top_missing_customers['Column'])
axes[0].set_xlabel('Null Percentage (%)')
axes[0].set_title('Top 10 Incomplete Fields - Customers', fontweight='bold')
axes[0].invert_yaxis()
axes[0].grid(axis='x', alpha=0.3)

# Transactions
top_missing_transactions = transactions_completeness.nlargest(10, 'Null %')
axes[1].barh(range(len(top_missing_transactions)), top_missing_transactions['Null %'], color='#D72638', alpha=0.7)
axes[1].set_yticks(range(len(top_missing_transactions)))
axes[1].set_yticklabels(top_missing_transactions['Column'])
axes[1].set_xlabel('Null Percentage (%)')
axes[1].set_title('Top 10 Incomplete Fields - Transactions', fontweight='bold')
axes[1].invert_yaxis()
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Duplicate Detection

In [ ]:
# Check for duplicate customers
customers = get_table_df('customers')

# Exact duplicates based on unique identifiers
duplicate_emails = DataQualityChecker.check_duplicates(customers, ['email'])
duplicate_ssn = DataQualityChecker.check_duplicates(customers, ['ssn'])
duplicate_cards = DataQualityChecker.check_duplicates(customers, ['card_number'])

print("DUPLICATE DETECTION - CUSTOMERS")
print("=" * 70)
print(f"Duplicate emails: {len(duplicate_emails)} rows")
print(f"Duplicate SSNs: {len(duplicate_ssn)} rows")
print(f"Duplicate card numbers: {len(duplicate_cards)} rows")

if len(duplicate_emails) > 0:
    print("\nSample duplicate emails:")
    print(duplicate_emails[['customer_id', 'email', 'first_name', 'last_name']].head(10))

# Near-duplicates (fuzzy matching on name)
# Group by similar names
customers['full_name'] = customers['first_name'] + ' ' + customers['last_name']
name_counts = customers['full_name'].value_counts()
duplicate_names = name_counts[name_counts > 1]

print(f"\nCustomers with identical names: {duplicate_names.sum()} (across {len(duplicate_names)} names)")
if len(duplicate_names) > 0:
    print("\nTop duplicate names:")
    print(duplicate_names.head(10))

print("=" * 70)

In [ ]:
# Check for duplicate transactions
transactions = get_table_df('transactions')

# Suspicious: Same card, same amount, same merchant, within 1 second
# (Could be legitimate or could be duplicate submission)
transactions_sorted = transactions.sort_values(['card_number', 'amount', 'created_date'])
transactions_sorted['time_diff'] = transactions_sorted.groupby(['card_number', 'amount'])['created_date'].diff()

# Potential duplicates (within 10 seconds)
potential_dupes = transactions_sorted[
    (transactions_sorted['time_diff'].notna()) & 
    (pd.to_timedelta(transactions_sorted['time_diff']).dt.total_seconds() < 10)
]

print("DUPLICATE DETECTION - TRANSACTIONS")
print("=" * 70)
print(f"Potential duplicate transactions (same card/amount within 10s): {len(potential_dupes)}")
print(f"Percentage of total: {len(potential_dupes)/len(transactions)*100:.2f}%")

if len(potential_dupes) > 0:
    print("\nSample potential duplicates:")
    print(potential_dupes[['id', 'card_number', 'amount', 'created_date', 'time_diff']].head(10))

print("=" * 70)

## 4. Referential Integrity Checks

In [ ]:
# Check for orphaned records
# Transactions should reference existing customers
orphaned_transactions = DataQualityChecker.check_referential_integrity(
    transactions, customers, 'customer_id', 'customer_id'
)

print("REFERENTIAL INTEGRITY CHECKS")
print("=" * 70)
print(f"Orphaned transactions (customer doesn't exist): {len(orphaned_transactions)}")

if len(orphaned_transactions) > 0:
    print("\nSample orphaned transactions:")
    print(orphaned_transactions[['id', 'customer_id', 'amount', 'created_date']].head(10))

# Check merchants
merchants = get_table_df('merchants')
orphaned_txn_merchants = DataQualityChecker.check_referential_integrity(
    transactions, merchants, 'merchant_id', 'merchant_id'
)

print(f"\nOrphaned transactions (merchant doesn't exist): {len(orphaned_txn_merchants)}")

print("=" * 70)

## 5. Data Validity Checks

In [ ]:
# Validate data formats and business rules

print("DATA VALIDITY CHECKS")
print("=" * 70)

# Email format validation
email_pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
invalid_emails = customers[~customers['email'].str.match(email_pattern, na=False)]
print(f"Invalid email formats: {len(invalid_emails)}")

# Phone number validation (US format)
phone_pattern = r'^\(?\d{3}\)?[-\s]?\d{3}[-\s]?\d{4}$'
invalid_phones = customers[
    (customers['phone'].notna()) & 
    (~customers['phone'].str.match(phone_pattern, na=False))
]
print(f"Invalid phone formats: {len(invalid_phones)}")

# SSN validation (XXX-XX-XXXX)
ssn_pattern = r'^\d{3}-\d{2}-\d{4}$'
invalid_ssn = customers[
    (customers['ssn'].notna()) & 
    (~customers['ssn'].str.match(ssn_pattern, na=False))
]
print(f"Invalid SSN formats: {len(invalid_ssn)}")

# Card expiry validation (should be future dates)
# Assuming MM/YY format
def is_card_expired(expiry_str):
    if pd.isna(expiry_str):
        return False
    try:
        month, year = expiry_str.split('/')
        expiry_date = datetime(2000 + int(year), int(month), 1)
        return expiry_date < datetime.now()
    except:
        return False

customers['card_expired'] = customers['card_expiry'].apply(is_card_expired)
expired_cards = customers[customers['card_expired']]
print(f"Expired cards: {len(expired_cards)}")

# Transaction amount validation (should be positive)
negative_amounts = transactions[transactions['amount'] <= 0]
print(f"\nNegative or zero transaction amounts: {len(negative_amounts)}")

# Extreme amounts (likely errors)
extreme_high = transactions[transactions['amount'] > 100000]  # >$100k
extreme_low = transactions[(transactions['amount'] > 0) & (transactions['amount'] < 0.01)]  # <1 cent
print(f"Extremely high amounts (>$100k): {len(extreme_high)}")
print(f"Extremely low amounts (<$0.01): {len(extreme_low)}")

# Future dates (data entry errors)
future_transactions = transactions[pd.to_datetime(transactions['created_date']) > datetime.now()]
print(f"Future-dated transactions: {len(future_transactions)}")

print("=" * 70)

## 6. Consistency Checks

In [ ]:
# Check for inconsistencies across related fields

print("CONSISTENCY CHECKS")
print("=" * 70)

# Customer email in transactions should match customer table
merged = transactions.merge(customers[['customer_id', 'email']], on='customer_id', how='left', suffixes=('_txn', '_cust'))
email_mismatches = merged[
    (merged['customer_email'].notna()) & 
    (merged['email'].notna()) & 
    (merged['customer_email'] != merged['email'])
]
print(f"Email inconsistencies (transaction vs customer): {len(email_mismatches)}")

# Check SSN consistency
merged_ssn = transactions.merge(customers[['customer_id', 'ssn']], on='customer_id', how='left', suffixes=('_txn', '_cust'))
ssn_mismatches = merged_ssn[
    (merged_ssn['customer_ssn'].notna()) & 
    (merged_ssn['ssn'].notna()) & 
    (merged_ssn['customer_ssn'] != merged_ssn['ssn'])
]
print(f"SSN inconsistencies: {len(ssn_mismatches)}")

# Status consistency (declined transactions shouldn't have authorization codes)
declined_with_auth = transactions[
    (transactions['status'] == 2) &  # Declined
    (transactions['authorization_code'].notna())
]
print(f"Declined transactions with authorization codes: {len(declined_with_auth)}")

# Approved transactions should have auth codes
approved_no_auth = transactions[
    (transactions['status'] == 1) &  # Approved
    (transactions['authorization_code'].isna())
]
print(f"Approved transactions without authorization codes: {len(approved_no_auth)}")

print("=" * 70)

## 7. Data Quality Scorecard

In [ ]:
# Calculate overall data quality scores

def calculate_dq_score(table_name, critical_fields=None):
    """Calculate data quality score for a table."""
    df = get_table_df(table_name)
    
    # Completeness score
    if critical_fields:
        completeness = (1 - df[critical_fields].isnull().sum().sum() / (len(df) * len(critical_fields))) * 100
    else:
        completeness = (1 - df.isnull().sum().sum() / (len(df) * len(df.columns))) * 100
    
    # Uniqueness score (based on key fields)
    # For this demo, use row count vs unique count
    uniqueness = 100  # Simplified for demo
    
    # Validity score (simplified)
    validity = 95  # Based on checks above
    
    # Overall score (weighted average)
    overall = (completeness * 0.4) + (uniqueness * 0.3) + (validity * 0.3)
    
    return {
        'Completeness': round(completeness, 1),
        'Uniqueness': round(uniqueness, 1),
        'Validity': round(validity, 1),
        'Overall': round(overall, 1)
    }

# Score main tables
customer_score = calculate_dq_score('customers', ['customer_id', 'email', 'first_name', 'last_name'])
transaction_score = calculate_dq_score('transactions', ['id', 'customer_id', 'amount', 'status'])
merchant_score = calculate_dq_score('merchants', ['merchant_id', 'name', 'api_key'])

scorecard = pd.DataFrame({
    'Table': ['Customers', 'Transactions', 'Merchants'],
    'Completeness': [customer_score['Completeness'], transaction_score['Completeness'], merchant_score['Completeness']],
    'Uniqueness': [customer_score['Uniqueness'], transaction_score['Uniqueness'], merchant_score['Uniqueness']],
    'Validity': [customer_score['Validity'], transaction_score['Validity'], merchant_score['Validity']],
    'Overall Score': [customer_score['Overall'], transaction_score['Overall'], merchant_score['Overall']]
})

print("DATA QUALITY SCORECARD")
print("=" * 70)
print(scorecard.to_string(index=False))
print("\nScoring: 90-100 = Excellent, 80-89 = Good, 70-79 = Fair, <70 = Poor")
print("=" * 70)

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(scorecard))
width = 0.2

ax.bar(x - width*1.5, scorecard['Completeness'], width, label='Completeness', alpha=0.8)
ax.bar(x - width*0.5, scorecard['Uniqueness'], width, label='Uniqueness', alpha=0.8)
ax.bar(x + width*0.5, scorecard['Validity'], width, label='Validity', alpha=0.8)
ax.bar(x + width*1.5, scorecard['Overall Score'], width, label='Overall', alpha=0.8, color='#2E86AB')

ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('Data Quality Scorecard by Dimension', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(scorecard['Table'])
ax.legend()
ax.set_ylim(0, 105)
ax.grid(axis='y', alpha=0.3)

# Add threshold line at 80%
ax.axhline(y=80, color='r', linestyle='--', linewidth=1, alpha=0.5, label='Target (80%)')

plt.tight_layout()
plt.show()

## 8. Data Quality Issues Summary

In [ ]:
# Compile all issues
issues = [
    {'Severity': 'Critical', 'Issue': 'Orphaned transactions', 'Count': len(orphaned_transactions), 'Impact': 'Revenue reporting inaccuracy'},
    {'Severity': 'High', 'Issue': 'Duplicate customers (email)', 'Count': len(duplicate_emails), 'Impact': 'Customer experience, data integrity'},
    {'Severity': 'High', 'Issue': 'Expired cards active', 'Count': len(expired_cards), 'Impact': 'Failed transactions, poor UX'},
    {'Severity': 'High', 'Issue': 'Email/SSN inconsistencies', 'Count': len(email_mismatches) + len(ssn_mismatches), 'Impact': 'Data trust, compliance risk'},
    {'Severity': 'Medium', 'Issue': 'Invalid email formats', 'Count': len(invalid_emails), 'Impact': 'Communication failures'},
    {'Severity': 'Medium', 'Issue': 'Invalid phone formats', 'Count': len(invalid_phones), 'Impact': 'Contact issues'},
    {'Severity': 'Medium', 'Issue': 'Potential duplicate transactions', 'Count': len(potential_dupes), 'Impact': 'Revenue overstatement'},
    {'Severity': 'Low', 'Issue': 'Invalid SSN formats', 'Count': len(invalid_ssn), 'Impact': 'Compliance documentation'},
]

issues_df = pd.DataFrame(issues).sort_values(['Severity', 'Count'], ascending=[True, False])

print("DATA QUALITY ISSUES SUMMARY")
print("=" * 70)
print(issues_df.to_string(index=False))
print("=" * 70)

# Priority issues for remediation
critical_high = issues_df[issues_df['Severity'].isin(['Critical', 'High'])]
total_critical_issues = critical_high['Count'].sum()

print(f"\n⚠️  TOTAL CRITICAL/HIGH ISSUES: {total_critical_issues:,}")
print("\nIMMEDIATE ACTIONS REQUIRED:")
for _, row in critical_high.iterrows():
    print(f"  • {row['Issue']}: {row['Count']:,} ({row['Impact']})")

## 9. Recommendations & Action Plan

### Key Findings:

**Data Quality Status:**
- Overall data quality: **Good** (80-90% range)
- Critical issues identified that need immediate attention
- Some systemic data entry problems detected

**Top Issues:**
1. Orphaned transactions (referential integrity)
2. Duplicate customer records
3. Expired payment methods
4. Data consistency across systems

**Business Impact:**
- Revenue reporting may be inaccurate
- Customer experience degraded by duplicate accounts
- Failed transactions due to expired cards
- Compliance risks from data inconsistencies

### Action Plan:

**Week 1 (Critical):**
- Investigate and resolve orphaned transactions
- Implement duplicate detection in customer onboarding
- Flag expired cards for customer update
- Add data validation rules at entry points

**Month 1 (High Priority):**
- Deploy automated data quality monitoring
- Create data quality dashboards for ongoing tracking
- Implement data validation rules in application layer
- Set up alerts for quality threshold breaches

**Quarter 1 (Important):**
- Establish data governance policies
- Define data quality SLAs
- Create data steward roles and responsibilities
- Implement master data management (MDM) for customers

**Ongoing:**
- Weekly data quality reports
- Monthly quality review meetings
- Continuous improvement of validation rules
- Regular audits of critical data elements

### Tools & Automation:

**Recommended Tools:**
- Great Expectations for data quality testing
- dbt for data transformations and testing
- Airflow for data quality pipeline orchestration
- Custom dashboards for quality monitoring

**Automation Opportunities:**
- Automated duplicate detection and merging
- Real-time validation at data entry
- Scheduled quality checks and alerting
- Self-healing data processes where possible